# Marine Species Detector — v3 Training (YOLOv8m)

**v2 baseline:** mAP@50=0.767, mAP@50-95=0.519 (YOLOv8s, 44 epochs, M1 Pro)

**v3 changes:** YOLOv8m (25M params), imgsz=896, 3265 images, Ptereleotris/Ophiuroidea uncapped, copy_paste=0.5, 150 epochs

**Set runtime to A100 GPU before running.**

In [ ]:
!nvidia-smi
import torch
print(f'CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
!pip install -q ultralytics fathomnet tqdm scikit-learn pandas Pillow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_DIR = '/content/drive/MyDrive/ocean-species-detector'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Saving to: {DRIVE_DIR}')

## Download FathomNet dataset (~15-30 min)

In [ ]:
import io, json, shutil, requests
import pandas as pd
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm
from PIL import Image
from sklearn.model_selection import train_test_split
from fathomnet.api import images as fn_images

DATA_DIR   = Path('/content/data')
IMAGES_DIR = DATA_DIR / 'images'
LABELS_DIR = DATA_DIR / 'labels'
RAW_DIR    = DATA_DIR / 'raw'

for split in ['train', 'val', 'test']:
    (IMAGES_DIR / split).mkdir(parents=True, exist_ok=True)
    (LABELS_DIR / split).mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(exist_ok=True)

SPECIES = [
    'Lutjanus campechanus',
    'Stenotomus caprinus',
    'Rhomboplites aurorubens',
    'Strongylocentrotus fragilis',
    'Ptereleotris',
    'Pagrus pagrus',
    'Chromis',
    'Epinephelus morio',
    'Ophiuroidea',
    'Balistes capriscus'
]

PER_SPECIES_CAP = {
    'Lutjanus campechanus': 600,
    'Stenotomus caprinus': 600,
    'Rhomboplites aurorubens': 600,
    'Strongylocentrotus fragilis': 600,
    'Ptereleotris': None,
    'Pagrus pagrus': 600,
    'Chromis': 600,
    'Epinephelus morio': 600,
    'Ophiuroidea': None,
    'Balistes capriscus': 600
}

CLASS_TO_ID = {s: i for i, s in enumerate(SPECIES)}

def download_image(url, dest):
    try:
        resp = requests.get(url, timeout=30, stream=True)
        resp.raise_for_status()
        img = Image.open(io.BytesIO(resp.content)).convert('RGB')
        img.save(dest, 'JPEG', quality=95)
        return True
    except Exception:
        return False

def bbox_to_yolo(x, y, w, h, img_w, img_h):
    xc = max(0., min(1., (x + w/2) / img_w))
    yc = max(0., min(1., (y + h/2) / img_h))
    wn = max(0., min(1., w / img_w))
    hn = max(0., min(1., h / img_h))
    return xc, yc, wn, hn

records = []
seen_uuids = set()
class_counts = defaultdict(int)

for concept in SPECIES:
    cap = PER_SPECIES_CAP[concept]
    print(f'Fetching: {concept} (cap={cap or "ALL"})')
    try:
        img_records = fn_images.find_by_concept(concept)
    except Exception as e:
        print(f'  ERROR: {e}')
        continue
    if cap:
        img_records = img_records[:cap]
    print(f'  {len(img_records)} images available')

    downloaded = 0
    for item in tqdm(img_records):
        uuid, url = item.uuid, item.url
        if not url or uuid in seen_uuids:
            continue
        seen_uuids.add(uuid)
        img_w, img_h = item.width, item.height
        if not img_w or not img_h:
            continue
        img_path = RAW_DIR / f'{uuid}.jpg'
        if not img_path.exists():
            if not download_image(url, img_path):
                continue
        yolo_lines = []
        for box in (item.boundingBoxes or []):
            if box.concept not in CLASS_TO_ID or box.width <= 0 or box.height <= 0:
                continue
            cid = CLASS_TO_ID[box.concept]
            xc, yc, wn, hn = bbox_to_yolo(box.x, box.y, box.width, box.height, img_w, img_h)
            yolo_lines.append(f'{cid} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}')
            class_counts[box.concept] += 1
        if yolo_lines:
            records.append({'uuid': uuid, 'img_path': str(img_path), 'annotations': '\n'.join(yolo_lines)})
            downloaded += 1
    print(f'  Collected: {downloaded}')

df = pd.DataFrame(records)
train_val, test = train_test_split(df, test_size=0.15, random_state=42)
train, val = train_test_split(train_val, test_size=0.176, random_state=42)
print(f'Split: train={len(train)}, val={len(val)}, test={len(test)}')

for split_name, split_df in [('train', train), ('val', val), ('test', test)]:
    for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc=split_name):
        src = Path(row['img_path'])
        if not src.exists(): continue
        shutil.copy2(src, IMAGES_DIR / split_name / f"{row['uuid']}.jpg")
        (LABELS_DIR / split_name / f"{row['uuid']}.txt").write_text(row['annotations'])

names_block = '\n'.join(f'  {i}: {s}' for i, s in enumerate(SPECIES))
(DATA_DIR / 'dataset.yaml').write_text(
    f'path: /content/data\ntrain: images/train\nval: images/val\ntest: images/test\nnc: {len(SPECIES)}\nnames:\n{names_block}\n'
)
json.dump({str(i): s for i, s in enumerate(SPECIES)}, open(DATA_DIR / 'class_map.json', 'w'), indent=2)

print('\nAnnotation counts:')
for s in SPECIES:
    print(f'  {s:<40} {class_counts[s]:>6} boxes')

## Train YOLOv8m

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8m.pt')
print(f'Parameters: {sum(p.numel() for p in model.model.parameters()):,}')

results = model.train(
    data='/content/data/dataset.yaml',
    project='/content/drive/MyDrive/ocean-species-detector/models',
    name='fathomnet_v3',
    device='cuda',
    epochs=150,
    batch=16,
    imgsz=896,
    patience=40,
    save_period=10,
    optimizer='AdamW',
    lr0=0.0008,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=5,
    cos_lr=True,
    freeze=5,
    box=7.5,
    cls=0.7,
    dfl=1.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.45,
    flipud=0.3,
    fliplr=0.5,
    degrees=10,
    scale=0.7,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.5,
    erasing=0.4,
    verbose=True,
    plots=True
)

print(f"mAP@50:    {results.results_dict.get('metrics/mAP50(B)', 'N/A')}")
print(f"mAP@50-95: {results.results_dict.get('metrics/mAP50-95(B)', 'N/A')}")

## Evaluate on test set

In [ ]:
import json
from ultralytics import YOLO

model = YOLO('/content/drive/MyDrive/ocean-species-detector/models/fathomnet_v3/weights/best.pt')
class_map = json.load(open('/content/data/class_map.json'))

results = model.val(data='/content/data/dataset.yaml', split='test', conf=0.25, iou=0.5, plots=True)

print(f'{"Class":<35} {"AP@50":>8} {"AP@50-95":>10} {"Recall":>8}')
print('-' * 65)
for i, cls_idx in enumerate(results.ap_class_index):
    name = class_map.get(str(cls_idx), f'class_{cls_idx}')
    print(f'{name:<35} {results.box.ap50[i]:>8.3f} {results.box.ap[i]:>10.3f} {results.box.r[i]:>8.3f}')
print('-' * 65)
print(f'{"mAP (all)":<35} {results.box.map50:>8.3f} {results.box.map:>10.3f}')
print(f'\nResume metric: mAP@50-95 = {results.box.map:.3f}')